### 📈 Stock Picking is Worse than Gambling at a Casino

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Apply to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### The Market v.s. Your Stock Pick

Tech always rips, it's not hard to "beat the market" and pick the next best stock, right?

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_vrt_daily_returns_10y.csv"

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"

SEED = 42
INITIAL_INDEX_VALUE = 100

HOT_STOCK_NAME = "Hot Stock"

# High-beta synthetic stock settings
BETA_TO_SPY = 1.35
IDIOSYNCRATIC_DAILY_VOL = 0.006

# 0.20 means Hot Stock ends with 20% more wealth than SPY.
# Example: SPY grows from 100 to 120; Hot Stock targets 144.
TARGET_RELATIVE_OUTPERFORMANCE = 0.20

# Animation settings
FRAME_STRIDE = 5
FRAME_DURATION = 20

# ============================================================
# Load and prepare SPY data
# ============================================================

df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()
df = df.reset_index(drop=True)

if df.empty:
    raise ValueError("No rows found in the selected date range.")

if "spy_close" not in df.columns:
    raise ValueError("CSV must include a 'spy_close' column.")

# Recompute returns inside the selected window so cumulative return starts cleanly at 0%.
df["spy_simple_return"] = df["spy_close"].pct_change().fillna(0)
df["spy_log_return"] = np.log1p(df["spy_simple_return"])

df["spy_index"] = INITIAL_INDEX_VALUE * np.exp(df["spy_log_return"].cumsum())
df["spy_cum_return"] = df["spy_index"] / INITIAL_INDEX_VALUE - 1

# ============================================================
# Generate synthetic high-beta Hot Stock path
# ============================================================

rng = np.random.default_rng(SEED)

n_days = len(df)

if n_days < 2:
    raise ValueError("Need at least two rows to create an animated return path.")

spy_log_returns = df["spy_log_return"].to_numpy()

# Start with a high-beta version of SPY plus some stock-specific noise.
noise = rng.normal(
    loc=0,
    scale=IDIOSYNCRATIC_DAILY_VOL,
    size=n_days
)

noise[0] = 0

raw_hot_log_returns = BETA_TO_SPY * spy_log_returns + noise
raw_hot_log_returns[0] = 0

# Force the Hot Stock to finish 20% ahead of SPY on a relative wealth basis.
spy_total_log_return = spy_log_returns.sum()
target_hot_total_log_return = spy_total_log_return + np.log1p(TARGET_RELATIVE_OUTPERFORMANCE)

alpha_adjustment_per_day = (
    target_hot_total_log_return - raw_hot_log_returns.sum()
) / (n_days - 1)

hot_log_returns = raw_hot_log_returns.copy()
hot_log_returns[1:] += alpha_adjustment_per_day

df["hot_stock_log_return"] = hot_log_returns
df["hot_stock_simple_return"] = np.expm1(df["hot_stock_log_return"])

df["hot_stock_index"] = INITIAL_INDEX_VALUE * np.exp(df["hot_stock_log_return"].cumsum())
df["hot_stock_cum_return"] = df["hot_stock_index"] / INITIAL_INDEX_VALUE - 1

# Optional: synthetic close column, normalized to 100.
df["hot_stock_close"] = df["hot_stock_index"]

# ============================================================
# Summary stats
# ============================================================

latest_date = df["date"].iloc[-1]

spy_final_return = df["spy_cum_return"].iloc[-1]
hot_final_return = df["hot_stock_cum_return"].iloc[-1]

relative_outperformance = (
    df["hot_stock_index"].iloc[-1] / df["spy_index"].iloc[-1] - 1
)

simple_outperformance = hot_final_return - spy_final_return

realized_corr = df["spy_simple_return"].corr(df["hot_stock_simple_return"])

# --- Compute extra stats for legendless titles ---
def annualized_cagr(cum_return, n_days):
    years = n_days / 252  # trading days
    return (1 + cum_return) ** (1 / years) - 1

def annualized_sharpe(simple_returns):
    if len(simple_returns) < 2:
        return np.nan
    mean = np.mean(simple_returns)
    std = np.std(simple_returns, ddof=1)
    if std == 0:
        return np.nan
    sharpe = mean / std * np.sqrt(252)
    return sharpe

def max_drawdown_from_cum(cum_returns):
    # cum_returns should be cumulative return, e.g. 0.18, -0.05, etc.
    wealth = (1 + cum_returns)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)

spy_cagr = annualized_cagr(spy_final_return, len(df))
hot_cagr = annualized_cagr(hot_final_return, len(df))
spy_sharpe = annualized_sharpe(df["spy_simple_return"])
hot_sharpe = annualized_sharpe(df["hot_stock_simple_return"])
spy_mdd = max_drawdown_from_cum(df["spy_cum_return"])
hot_mdd = max_drawdown_from_cum(df["hot_stock_cum_return"])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
spy_color = "#00ff88"
hot_stock_color = "#ffaa33"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
y_min = min(df["spy_cum_return"].min(), df["hot_stock_cum_return"].min())
y_max = max(df["spy_cum_return"].max(), df["hot_stock_cum_return"].max())
y_pad = max((y_max - y_min) * 0.12, 0.03)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        f"SPY: CAGR {spy_cagr:.2%} | Sharpe {spy_sharpe:.2f} | MDD {spy_mdd:.1%}",
        f"{HOT_STOCK_NAME}: CAGR {hot_cagr:.2%} | Sharpe {hot_sharpe:.2f} | MDD {hot_mdd:.1%}"
    )
)

# Zero lines
fig.add_hline(
    y=0,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=0,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Left: SPY cumulative return
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["spy_cum_return"].iloc[0]],
        mode="lines",
        line=dict(color=spy_color, width=4),
        name="SPY",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "SPY Cumulative Return: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# The following right subplots will only show the Hot Stock.
# (Remove SPY Benchmark on right chart)
# Right: Hot Stock cumulative return
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["hot_stock_cum_return"].iloc[0]],
        mode="lines",
        line=dict(color=hot_stock_color, width=4),
        name=HOT_STOCK_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{HOT_STOCK_NAME} Cumulative Return: "
            "%{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["spy_cum_return"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["hot_stock_cum_return"]
                )
            ],
            traces=[0, 1],  # adjust number of traces due to legendless right
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            f"SPY vs. Synthetic High-Beta Hot Stock"
            f"<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"SPY Return: {spy_final_return:.1%} · "
            f"{HOT_STOCK_NAME} Return: {hot_final_return:.1%} · "
            f"Relative Beat: {relative_outperformance:.1%} · "
            f"Return Spread: {simple_outperformance:.1%} · "
            f"Daily Return Corr: {realized_corr:.2f} · "
            f"Beta Input: {BETA_TO_SPY:.2f}"
            f"</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,  # REMOVE LEGEND
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[y_min - y_pad, y_max + y_pad],
    title_text="SPY Cumulative Return",
    tickformat=".0%"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[y_min - y_pad, y_max + y_pad],
    title_text="Cumulative Return",
    tickformat=".0%"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("spy_vs_hot_stock_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

**Problem:** Even if you're picks are *always right* there's a big problem

A significant portion of your return is driven by simple beta exposure, something you can never control.

Let's suppose you were a great stock picker and found the next hot AI stock: VRT.

Even if you were right about the company, your portfolio might blowout before you see the gains of your pick.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_vrt_daily_returns_10y.csv"

START_DATE = "2025-01-01"
END_DATE = "2025-07-31"   # Only go to July 2025

INITIAL_INDEX_VALUE = 100

FRAME_STRIDE = 5
FRAME_DURATION = 20

RISK_FREE_RATE = 0.02  # For Sharpe ratio, 2% annualized

# ============================================================
# Load and prepare data
# ============================================================

df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

required_cols = ["date", "spy_close", "vrt_close"]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing required columns: {missing_cols}")

df = df[
    (df["date"] >= START_DATE) &
    (df["date"] <= END_DATE)
].copy()

# Keep only dates where both SPY and VRT have prices
df = df.dropna(subset=["spy_close", "vrt_close"]).reset_index(drop=True)

if len(df) < 2:
    raise ValueError("Not enough overlapping SPY/VRT rows in the selected date range.")

# Recompute returns within selected window so both start cleanly at 0%
df["spy_simple_return"] = df["spy_close"].pct_change().fillna(0)
df["vrt_simple_return"] = df["vrt_close"].pct_change().fillna(0)

df["spy_log_return"] = np.log1p(df["spy_simple_return"])
df["vrt_log_return"] = np.log1p(df["vrt_simple_return"])

# Normalize both to 100 for apples-to-apples return comparison
df["spy_index"] = INITIAL_INDEX_VALUE * np.exp(df["spy_log_return"].cumsum())
df["vrt_index"] = INITIAL_INDEX_VALUE * np.exp(df["vrt_log_return"].cumsum())

df["spy_cum_return"] = df["spy_index"] / INITIAL_INDEX_VALUE - 1
df["vrt_cum_return"] = df["vrt_index"] / INITIAL_INDEX_VALUE - 1

# ============================================================
# Summary stats & Metrics
# ============================================================

latest_date = df["date"].iloc[-1]

spy_final_return = df["spy_cum_return"].iloc[-1]
vrt_final_return = df["vrt_cum_return"].iloc[-1]

return_spread = vrt_final_return - spy_final_return
relative_outperformance = df["vrt_index"].iloc[-1] / df["spy_index"].iloc[-1] - 1

realized_corr = df["spy_simple_return"].corr(df["vrt_simple_return"])

spy_vol = df["spy_simple_return"].std() * np.sqrt(252)
vrt_vol = df["vrt_simple_return"].std() * np.sqrt(252)

# Regression style realized beta of VRT to SPY (via simple regression for display)
reg = LinearRegression().fit(df["spy_simple_return"].values.reshape(-1, 1),
                             df["vrt_simple_return"].values)
vrt_beta_reg = reg.coef_[0]

# "Classic" realized beta directly as covariance/variance
spy_var = df["spy_simple_return"].var()
vrt_beta = (
    df["spy_simple_return"].cov(df["vrt_simple_return"]) / spy_var
    if spy_var != 0
    else np.nan
)

# Utility: Calculate CAGR, Sharpe, MDD for a given index and returns
def calc_cagr(final_value, initial_value, days):
    years = days / 252  # trading days per year
    if initial_value == 0 or final_value <= 0:
        return np.nan
    return (final_value / initial_value)**(1/years) - 1

def calc_sharpe(daily_returns, risk_free_rate=RISK_FREE_RATE):
    # risk_free_rate is annualized
    excess_daily = daily_returns - (risk_free_rate / 252)
    if excess_daily.std() == 0:
        return np.nan
    return np.sqrt(252) * excess_daily.mean() / excess_daily.std()

def calc_mdd(index_series):
    # Compute running max
    running_max = index_series.cummax()
    drawdown = (index_series - running_max) / running_max
    return drawdown.min()

n_days = (df["date"].iloc[-1] - df["date"].iloc[0]).days
n_periods = len(df)

spy_cagr = calc_cagr(df["spy_index"].iloc[-1], INITIAL_INDEX_VALUE, n_periods)
vrt_cagr = calc_cagr(df["vrt_index"].iloc[-1], INITIAL_INDEX_VALUE, n_periods)

spy_sharpe = calc_sharpe(df["spy_simple_return"])
vrt_sharpe = calc_sharpe(df["vrt_simple_return"])

spy_mdd = calc_mdd(df["spy_index"])
vrt_mdd = calc_mdd(df["vrt_index"])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
spy_color = "#00ff88"
vrt_color = "#ffaa33"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
y_min = min(df["spy_cum_return"].min(), df["vrt_cum_return"].min())
y_max = max(df["spy_cum_return"].max(), df["vrt_cum_return"].max())
y_pad = max((y_max - y_min) * 0.12, 0.03)

# ============================================================
# Subplot-specific metric strings
# ============================================================

# Rewritten subtitles per instructions!
spy_subtitle = (
    "<b>SPY</b><br>"
    f"CAGR: {spy_cagr:.1%}, Sharpe: {spy_sharpe:.2f}, MDD: {spy_mdd:.1%}, Beta: 1.00"
)

vrt_subtitle = (
    "<b>VRT</b><br>"
    f"CAGR: {vrt_cagr:.1%}, Sharpe: {vrt_sharpe:.2f}, MDD: {vrt_mdd:.1%}, Beta: {vrt_beta_reg:.2f}"
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(spy_subtitle, vrt_subtitle)
)

# Zero lines
fig.add_hline(
    y=0,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=0,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Left: SPY cumulative return
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["spy_cum_return"].iloc[0]],
        mode="lines",
        line=dict(color=spy_color, width=4),
        name="SPY",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "SPY cumulative return: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Right: SPY benchmark
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["spy_cum_return"].iloc[0]],
        mode="lines",
        line=dict(color=spy_color, width=3, dash="dash"),
        opacity=0.75,
        name="SPY Benchmark",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "SPY cumulative return: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Right: VRT cumulative return
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["vrt_cum_return"].iloc[0]],
        mode="lines",
        line=dict(color=vrt_color, width=4),
        name="VRT",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "VRT cumulative return: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["spy_cum_return"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["spy_cum_return"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["vrt_cum_return"]
                )
            ],
            traces=[0, 1, 2],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

# NON-metric info to add as global title (not metrics):
main_title_lines = [
    f"VRT - SPY Spread: {return_spread:.1%} · Relative Outperformance: {relative_outperformance:.1%}",
    f"{df['date'].iloc[0]:%Y-%m-%d} to {latest_date:%Y-%m-%d}"
]

fig.update_layout(
    title=dict(
        text="<br>".join(main_title_lines),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=110, b=150, r=50, l=75),  # Lower top margin since per-panel subtitle is now metrics
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[y_min - y_pad, y_max + y_pad],
    title_text="SPY Cumulative Return",
    tickformat=".0%"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[y_min - y_pad, y_max + y_pad],
    title_text="Cumulative Return",
    tickformat=".0%"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("spy_vs_vrt_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### That's a Cherry Picked Example

No it's not, it's a big f*cking problem for your stock picking.

Likely, most of the return is not idiosyncratic and is driven by extreme beta exposure.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_vrt_daily_returns_10y.csv"

TRADING_DAYS_PER_YEAR = 252
SIM_YEARS = 10
SIM_DAYS = SIM_YEARS * TRADING_DAYS_PER_YEAR

INITIAL_INDEX_VALUE = 100

N_SIM_PATHS = 5000

# Keep this modest for smooth animation.
# 8 to 12 is usually enough visually.
N_DISPLAY_PATHS = 8

SEED = 42

FRAME_STRIDE = 21
FRAME_DURATION = 30

RISK_FREE_RATE = 0.02

HOT_STOCK_BETA = 2.5
HOT_STOCK_NAME = "2.5 Beta Hot Stock"

TOP_Y_RANGE = [0, 1000]

FIT_START_DATE = None
FIT_END_DATE = None

USE_RISK_FREE_CAPM_DRIFT = True

# For speed. Set True only if your bars disappear in your local renderer.
ANIMATION_REDRAW = False

# ============================================================
# Load data
# ============================================================

df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

required_cols = ["date", "spy_close"]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing required columns: {missing_cols}")

fit_df = df.copy()

if FIT_START_DATE is not None:
    fit_df = fit_df[fit_df["date"] >= FIT_START_DATE]

if FIT_END_DATE is not None:
    fit_df = fit_df[fit_df["date"] <= FIT_END_DATE]

fit_df = fit_df.reset_index(drop=True)

# ============================================================
# GBM parameter estimation
# ============================================================

def fit_gbm_params_from_prices(data, price_col, trading_days_per_year=252):
    asset_df = data[["date", price_col]].dropna().copy()
    asset_df = asset_df.sort_values("date").reset_index(drop=True)

    if len(asset_df) < 3:
        raise ValueError(f"Not enough price observations for {price_col}")

    asset_df["simple_return"] = asset_df[price_col].pct_change()
    asset_df["log_return"] = np.log(asset_df[price_col] / asset_df[price_col].shift(1))

    returns_df = asset_df.dropna(subset=["simple_return", "log_return"]).copy()

    daily_log_mean = returns_df["log_return"].mean()
    daily_log_var = returns_df["log_return"].var(ddof=1)

    annual_log_growth = daily_log_mean * trading_days_per_year
    annual_variance = daily_log_var * trading_days_per_year
    annual_vol = np.sqrt(annual_variance)

    mu_annual = annual_log_growth + 0.5 * annual_variance
    vol_drag = 0.5 * annual_variance

    params = {
        "name": price_col,
        "price_col": price_col,
        "start_date": asset_df["date"].iloc[0],
        "end_date": asset_df["date"].iloc[-1],
        "n_obs": len(returns_df),

        "daily_log_mean": daily_log_mean,
        "daily_log_variance": daily_log_var,

        "mu_annual": mu_annual,
        "annual_log_growth": annual_log_growth,
        "annual_variance": annual_variance,
        "annual_vol": annual_vol,
        "vol_drag": vol_drag,

        "arithmetic_expected_annual_return": np.exp(mu_annual) - 1,
        "geometric_expected_annual_return": np.exp(annual_log_growth) - 1,
    }

    return params, returns_df


spy_params, spy_returns_df = fit_gbm_params_from_prices(
    fit_df,
    "spy_close",
    TRADING_DAYS_PER_YEAR
)

# ============================================================
# Generate 2.5 beta hot stock strategy from SPY parameters
# ============================================================

def make_beta_strategy_params(
    base_params,
    beta=2.5,
    risk_free_rate=0.02,
    use_capm_drift=True,
    name="2.5 Beta Hot Stock"
):
    spy_mu = base_params["mu_annual"]
    spy_sigma = base_params["annual_vol"]

    if use_capm_drift:
        hot_mu = risk_free_rate + beta * (spy_mu - risk_free_rate)
    else:
        hot_mu = beta * spy_mu

    hot_sigma = beta * spy_sigma
    hot_variance = hot_sigma ** 2
    hot_vol_drag = 0.5 * hot_variance
    hot_annual_log_growth = hot_mu - hot_vol_drag

    hot_params = {
        "name": name,
        "beta": beta,
        "start_date": base_params["start_date"],
        "end_date": base_params["end_date"],
        "n_obs": base_params["n_obs"],

        "mu_annual": hot_mu,
        "annual_log_growth": hot_annual_log_growth,
        "annual_variance": hot_variance,
        "annual_vol": hot_sigma,
        "vol_drag": hot_vol_drag,

        "arithmetic_expected_annual_return": np.exp(hot_mu) - 1,
        "geometric_expected_annual_return": np.exp(hot_annual_log_growth) - 1,
    }

    return hot_params


hot_params = make_beta_strategy_params(
    base_params=spy_params,
    beta=HOT_STOCK_BETA,
    risk_free_rate=RISK_FREE_RATE,
    use_capm_drift=USE_RISK_FREE_CAPM_DRIFT,
    name=HOT_STOCK_NAME
)

# ============================================================
# GBM simulation
# ============================================================

def simulate_gbm_paths(
    mu_annual,
    sigma_annual,
    years=10,
    trading_days_per_year=252,
    n_paths=5000,
    s0=100,
    seed=42
):
    rng = np.random.default_rng(seed)

    n_days = years * trading_days_per_year
    dt = 1 / trading_days_per_year

    z = rng.normal(size=(n_days, n_paths))

    log_returns = (
        (mu_annual - 0.5 * sigma_annual**2) * dt
        + sigma_annual * np.sqrt(dt) * z
    )

    log_paths = np.vstack([
        np.zeros(n_paths),
        np.cumsum(log_returns, axis=0)
    ])

    paths = s0 * np.exp(log_paths)

    return paths, log_returns


spy_paths, spy_sim_log_returns = simulate_gbm_paths(
    mu_annual=spy_params["mu_annual"],
    sigma_annual=spy_params["annual_vol"],
    years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    n_paths=N_SIM_PATHS,
    s0=INITIAL_INDEX_VALUE,
    seed=SEED
)

hot_paths, hot_sim_log_returns = simulate_gbm_paths(
    mu_annual=hot_params["mu_annual"],
    sigma_annual=hot_params["annual_vol"],
    years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    n_paths=N_SIM_PATHS,
    s0=INITIAL_INDEX_VALUE,
    seed=SEED + 1
)

last_data_date = spy_params["end_date"]

sim_dates = pd.bdate_range(
    start=last_data_date + pd.offsets.BDay(1),
    periods=SIM_DAYS + 1
)

t_years = np.arange(SIM_DAYS + 1) / TRADING_DAYS_PER_YEAR

# ============================================================
# Pick paths to display
# ============================================================

display_rng = np.random.default_rng(SEED + 999)

display_path_indices = display_rng.choice(
    np.arange(N_SIM_PATHS),
    size=N_DISPLAY_PATHS,
    replace=False
)

# ============================================================
# Simulation summaries
# ============================================================

def summarize_paths(paths):
    return {
        "p10": np.percentile(paths, 10, axis=1),
        "p50": np.percentile(paths, 50, axis=1),
        "p90": np.percentile(paths, 90, axis=1),
        "mean_mc": paths.mean(axis=1)
    }


spy_summary = summarize_paths(spy_paths)
hot_summary = summarize_paths(hot_paths)


def theoretical_paths(params, t_years, s0=100):
    mu = params["mu_annual"]
    sigma = params["annual_vol"]

    arithmetic_mean_path = s0 * np.exp(mu * t_years)
    geometric_median_path = s0 * np.exp((mu - 0.5 * sigma**2) * t_years)

    return arithmetic_mean_path, geometric_median_path


spy_mean_path, spy_geo_path = theoretical_paths(
    spy_params,
    t_years,
    INITIAL_INDEX_VALUE
)

hot_mean_path, hot_geo_path = theoretical_paths(
    hot_params,
    t_years,
    INITIAL_INDEX_VALUE
)

# ============================================================
# Statistics for fixed bottom bar charts
# ============================================================

def calc_path_mdds(paths):
    running_max = np.maximum.accumulate(paths, axis=0)
    drawdowns = paths / running_max - 1
    max_drawdowns = drawdowns.min(axis=0)

    return -max_drawdowns


def calc_simulation_stats(paths, years, s0=100):
    final_values = paths[-1, :]

    path_cagrs = (final_values / s0) ** (1 / years) - 1
    path_mdds = calc_path_mdds(paths)

    stats = {
        "avg_cagr": np.nanmean(path_cagrs),
        "avg_mdd": np.nanmean(path_mdds),
        "terminal_1pct_value": np.percentile(final_values, 1),
        "mean_final": np.nanmean(final_values),
        "median_final": np.nanmedian(final_values),
        "p10_final": np.percentile(final_values, 10),
        "p90_final": np.percentile(final_values, 90),
    }

    return stats


spy_stats = calc_simulation_stats(
    paths=spy_paths,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

hot_stats = calc_simulation_stats(
    paths=hot_paths,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

mean_color = "#33aaff"
geo_color = "#ff6666"

band_color_spy = "rgba(0,255,136,0.16)"
band_color_hot = "rgba(255,170,51,0.16)"

path_color_spy = "rgba(224,224,224,0.28)"
path_color_hot = "rgba(224,224,224,0.28)"

baseline_color = "#777777"

percent_bar_color = "#33aaff"
value_bar_color = "#ff6666"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# ============================================================
# Bottom chart data
# ============================================================

percent_stat_labels = [
    "Avg.<br>CAGR",
    "Avg.<br>MDD",
    "Volatility<br>Drag"
]

value_stat_labels = [
    "1%<br>Value"
]

def percent_values_from_stats(stats, params):
    return [
        stats["avg_cagr"] * 100,
        stats["avg_mdd"] * 100,
        params["vol_drag"] * 100
    ]

def value_values_from_stats(stats):
    return [
        stats["terminal_1pct_value"]
    ]

def percent_text(values):
    return [f"{v:.1f}%" for v in values]

def value_text(values):
    return [f"{v:.0f}" for v in values]

spy_percent_values = percent_values_from_stats(spy_stats, spy_params)
hot_percent_values = percent_values_from_stats(hot_stats, hot_params)

spy_value_values = value_values_from_stats(spy_stats)
hot_value_values = value_values_from_stats(hot_stats)

all_percent_values = spy_percent_values + hot_percent_values
all_value_values = spy_value_values + hot_value_values

percent_axis_min = min(0, min(all_percent_values) * 1.20)
percent_axis_max = max(10, max(all_percent_values) * 1.20)

value_axis_min = 0
value_axis_max = max(100, max(all_value_values) * 1.35)

# ============================================================
# Subplot titles
# ============================================================

def make_subplot_title(ticker, params, beta_text=None):
    beta_part = f" · Beta: {beta_text}" if beta_text is not None else ""

    return (
        f"<b>{ticker}</b>{beta_part}<br>"
        f"μ: {params['mu_annual']:.1%} · "
        f"σ: {params['annual_vol']:.1%} · "
        f"Vol drag: {params['vol_drag']:.1%}<br>"
        f"Geo growth: {params['annual_log_growth']:.1%}"
    )


spy_subtitle = make_subplot_title(
    "SPY",
    spy_params,
    beta_text="1.0"
)

hot_subtitle = make_subplot_title(
    HOT_STOCK_NAME,
    hot_params,
    beta_text=f"{HOT_STOCK_BETA:.1f}"
)

# ============================================================
# Figure
# ============================================================

# INCREASED VERTICAL SPACING BETWEEN BOTTOM TWO CHARTS!
fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.50, 0.50],
    # Increase bottom bar chart row height and vertical spacing for more gap between bottom plots
    row_heights=[0.65, 0.35],   # originally [0.68, 0.32]
    horizontal_spacing=0.10,
    vertical_spacing=0.275,     # originally 0.18 - increase for more gap!
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": True}, {"secondary_y": True}]
    ],
    subplot_titles=(
        spy_subtitle,
        hot_subtitle,
        "SPY Metrics",
        f"{HOT_STOCK_NAME} Metrics"
    )
)

# This stores only the animated top traces, in the same order they are added.
animated_y_series = []

def add_animated_trace(row, col, y_series, trace):
    fig.add_trace(trace, row=row, col=col)
    animated_y_series.append(y_series)

# ============================================================
# Top row: SPY simulation traces
# ... (unchanged code)
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

add_animated_trace(
    1,
    1,
    spy_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[spy_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name="SPY P90"
    )
)

add_animated_trace(
    1,
    1,
    spy_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[spy_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_spy,
        showlegend=False,
        hoverinfo="skip",
        name="SPY 10th-90th Percentile Band"
    )
)

for path_num, path_idx in enumerate(display_path_indices, start=1):
    y_path = spy_paths[:, path_idx]

    add_animated_trace(
        1,
        1,
        y_path,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_path[0]],
            mode="lines",
            line=dict(color=path_color_spy, width=1),
            opacity=0.85,
            showlegend=False,
            hoverinfo="skip",
            name=f"SPY Path {path_num}"
        )
    )

add_animated_trace(
    1,
    1,
    spy_mean_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[spy_mean_path[0]],
        mode="lines",
        line=dict(color=mean_color, width=4),
        showlegend=False,
        name="SPY Arithmetic Mean Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Arithmetic expected value: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    1,
    spy_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[spy_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name="SPY Geometric / Median Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Geometric / median value: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Top row: 2.5 beta hot stock simulation traces
# ... (unchanged code)
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

add_animated_trace(
    1,
    2,
    hot_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} P90"
    )
)

add_animated_trace(
    1,
    2,
    hot_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hot,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} 10th-90th Percentile Band"
    )
)

for path_num, path_idx in enumerate(display_path_indices, start=1):
    y_path = hot_paths[:, path_idx]

    add_animated_trace(
        1,
        2,
        y_path,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_path[0]],
            mode="lines",
            line=dict(color=path_color_hot, width=1),
            opacity=0.85,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HOT_STOCK_NAME} Path {path_num}"
        )
    )

add_animated_trace(
    1,
    2,
    hot_mean_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_mean_path[0]],
        mode="lines",
        line=dict(color=mean_color, width=4),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Arithmetic Mean Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Arithmetic expected value: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    2,
    hot_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Geometric / Median Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Geometric / median value: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Bottom row: fixed metric bar charts
# ... (unchanged code)
# ============================================================

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=spy_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(spy_percent_values),
        textposition="outside",
        showlegend=False,
        name="SPY Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=spy_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(spy_value_values),
        textposition="outside",
        showlegend=False,
        name="SPY 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=True
)

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hot_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hot_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hot_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hot_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=True
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

n_top_traces = len(animated_y_series)

frame_indices = list(range(1, len(sim_dates), FRAME_STRIDE))

if frame_indices[-1] != len(sim_dates) - 1:
    frame_indices.append(len(sim_dates) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    x_slice = sim_dates[:i + 1]

    frame_data = []

    for y_series in animated_y_series:
        frame_data.append(
            go.Scatter(
                x=x_slice,
                y=y_series[:i + 1]
            )
        )

    # Bottom bars are repeated in every frame so they remain visible.
    frame_data.extend([
        go.Bar(
            x=percent_stat_labels,
            y=spy_percent_values,
            text=percent_text(spy_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=spy_value_values,
            text=value_text(spy_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=percent_stat_labels,
            y=hot_percent_values,
            text=percent_text(hot_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hot_value_values,
            text=value_text(hot_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        )
    ])

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(n_top_traces + 4)),
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": sim_dates[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="SPY vs. 2.5-Beta Hot Stock Strategy: Volatility Drag and Long-Run Performance",
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=950,
    width=1200,
    margin=dict(t=155, b=150, r=80, l=75),
    showlegend=False,
    hovermode="closest",
    barmode="group",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": ANIMATION_REDRAW},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=13))

# ============================================================
# Axes: top row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=TOP_Y_RANGE,
    title_text="Simulated Value, Start = 100"
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=TOP_Y_RANGE,
    title_text="Simulated Value, Start = 100"
)

# ============================================================
# Axes: bottom row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Percent",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="",
    showgrid=False
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=2,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Percent",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="",
    showgrid=False
)

fig.update_traces(
    cliponaxis=False,
    selector=dict(type="bar")
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("spy_vs_2p5_beta_hot_stock_multi_paths_fast.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### You Don't Get Something for Nothing

Roman's version of "No Free Lunch".

If you want to operate effectively in the space you need to understand what you're doing.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_vrt_daily_returns_10y.csv"

TRADING_DAYS_PER_YEAR = 252
SIM_YEARS = 10
SIM_DAYS = SIM_YEARS * TRADING_DAYS_PER_YEAR

INITIAL_INDEX_VALUE = 100

N_SIM_PATHS = 5000
N_DISPLAY_PATHS = 8

SEED = 42

FRAME_STRIDE = 21
FRAME_DURATION = 30
ANIMATION_REDRAW = False

RISK_FREE_RATE = 0.02

HOT_STOCK_BETA = 2.5
HOT_STOCK_NAME = "2.5 Beta Hot Stock"
HEDGE_NAME = "Hedge"

BOTTOM_DECILE = 0.10

TOP_Y_RANGE = [0, 1000]

FIT_START_DATE = None
FIT_END_DATE = None

USE_RISK_FREE_CAPM_DRIFT = True

# ============================================================
# Hedge strategy parameters
# ============================================================

HEDGE_RESET_DAYS = 63
HEDGE_FLOOR_LOSS = 0.10
HEDGE_UPSIDE_CAP = 0.28
HEDGE_CARRY_ANNUAL = 0.00
HEDGE_REDEPLOY_TRIGGER = -0.05
HEDGE_REDEPLOY_RATE = 1.00

# ============================================================
# Load data
# ============================================================

df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

required_cols = ["date", "spy_close"]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing required columns: {missing_cols}")

fit_df = df.copy()

if FIT_START_DATE is not None:
    fit_df = fit_df[fit_df["date"] >= FIT_START_DATE]

if FIT_END_DATE is not None:
    fit_df = fit_df[fit_df["date"] <= FIT_END_DATE]

fit_df = fit_df.reset_index(drop=True)

# ============================================================
# GBM parameter estimation
# ============================================================

def fit_gbm_params_from_prices(data, price_col, trading_days_per_year=252):
    asset_df = data[["date", price_col]].dropna().copy()
    asset_df = asset_df.sort_values("date").reset_index(drop=True)

    if len(asset_df) < 3:
        raise ValueError(f"Not enough price observations for {price_col}")

    asset_df["simple_return"] = asset_df[price_col].pct_change()
    asset_df["log_return"] = np.log(asset_df[price_col] / asset_df[price_col].shift(1))

    returns_df = asset_df.dropna(subset=["simple_return", "log_return"]).copy()

    daily_log_mean = returns_df["log_return"].mean()
    daily_log_var = returns_df["log_return"].var(ddof=1)

    annual_log_growth = daily_log_mean * trading_days_per_year
    annual_variance = daily_log_var * trading_days_per_year
    annual_vol = np.sqrt(annual_variance)

    mu_annual = annual_log_growth + 0.5 * annual_variance
    vol_drag = 0.5 * annual_variance

    params = {
        "name": price_col,
        "price_col": price_col,
        "start_date": asset_df["date"].iloc[0],
        "end_date": asset_df["date"].iloc[-1],
        "n_obs": len(returns_df),

        "daily_log_mean": daily_log_mean,
        "daily_log_variance": daily_log_var,

        "mu_annual": mu_annual,
        "annual_log_growth": annual_log_growth,
        "annual_variance": annual_variance,
        "annual_vol": annual_vol,
        "vol_drag": vol_drag,

        "arithmetic_expected_annual_return": np.exp(mu_annual) - 1,
        "geometric_expected_annual_return": np.exp(annual_log_growth) - 1,
    }

    return params, returns_df


spy_params, spy_returns_df = fit_gbm_params_from_prices(
    fit_df,
    "spy_close",
    TRADING_DAYS_PER_YEAR
)

# ============================================================
# Generate 2.5 beta hot stock strategy from SPY parameters
# ============================================================

def make_beta_strategy_params(
    base_params,
    beta=2.5,
    risk_free_rate=0.02,
    use_capm_drift=True,
    name="2.5 Beta Hot Stock"
):
    spy_mu = base_params["mu_annual"]
    spy_sigma = base_params["annual_vol"]

    if use_capm_drift:
        hot_mu = risk_free_rate + beta * (spy_mu - risk_free_rate)
    else:
        hot_mu = beta * spy_mu

    hot_sigma = beta * spy_sigma
    hot_variance = hot_sigma ** 2
    hot_vol_drag = 0.5 * hot_variance
    hot_annual_log_growth = hot_mu - hot_vol_drag

    hot_params = {
        "name": name,
        "beta": beta,
        "start_date": base_params["start_date"],
        "end_date": base_params["end_date"],
        "n_obs": base_params["n_obs"],

        "mu_annual": hot_mu,
        "annual_log_growth": hot_annual_log_growth,
        "annual_variance": hot_variance,
        "annual_vol": hot_sigma,
        "vol_drag": hot_vol_drag,

        "arithmetic_expected_annual_return": np.exp(hot_mu) - 1,
        "geometric_expected_annual_return": np.exp(hot_annual_log_growth) - 1,
    }

    return hot_params


hot_params = make_beta_strategy_params(
    base_params=spy_params,
    beta=HOT_STOCK_BETA,
    risk_free_rate=RISK_FREE_RATE,
    use_capm_drift=USE_RISK_FREE_CAPM_DRIFT,
    name=HOT_STOCK_NAME
)

# ============================================================
# GBM simulation
# ============================================================

def simulate_gbm_paths(
    mu_annual,
    sigma_annual,
    years=10,
    trading_days_per_year=252,
    n_paths=5000,
    s0=100,
    seed=42
):
    rng = np.random.default_rng(seed)

    n_days = years * trading_days_per_year
    dt = 1 / trading_days_per_year

    z = rng.normal(size=(n_days, n_paths))

    log_returns = (
        (mu_annual - 0.5 * sigma_annual**2) * dt
        + sigma_annual * np.sqrt(dt) * z
    )

    log_paths = np.vstack([
        np.zeros(n_paths),
        np.cumsum(log_returns, axis=0)
    ])

    paths = s0 * np.exp(log_paths)

    return paths, log_returns


hot_paths, hot_sim_log_returns = simulate_gbm_paths(
    mu_annual=hot_params["mu_annual"],
    sigma_annual=hot_params["annual_vol"],
    years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    n_paths=N_SIM_PATHS,
    s0=INITIAL_INDEX_VALUE,
    seed=SEED + 1
)

last_data_date = spy_params["end_date"]

sim_dates = pd.bdate_range(
    start=last_data_date + pd.offsets.BDay(1),
    periods=SIM_DAYS + 1
)

t_years = np.arange(SIM_DAYS + 1) / TRADING_DAYS_PER_YEAR

# ============================================================
# Hedge strategy simulation
# ============================================================

def simulate_hedge_strategy_from_underlying(
    underlying_paths,
    reset_days=63,
    floor_loss=0.10,
    upside_cap=0.28,
    carry_annual=0.00,
    redeploy_trigger=-0.05,
    redeploy_rate=1.00,
    trading_days_per_year=252,
    s0=100
):
    """
    Systematic Hedge framework:

    1. Start with exposure to the 2.5-beta underlying.
    2. At every reset date, define a downside floor and an upside give-up level.
    3. During large selloffs, the downside floor creates monetizable cash.
    4. When a reset period ends below the redeploy trigger,
       positive hedge cash is used to buy more shares.
    5. During strong rallies, gains above the upside level are given up.
    """

    n_steps, n_paths = underlying_paths.shape
    strategy_values = np.zeros_like(underlying_paths)

    for path_idx in range(n_paths):
        shares = s0 / underlying_paths[0, path_idx]
        cash = 0.0

        strategy_values[0, path_idx] = s0

        start = 0

        while start < n_steps - 1:
            end = min(start + reset_days, n_steps - 1)

            start_price = underlying_paths[start, path_idx]
            floor_strike = start_price * (1 - floor_loss)
            cap_strike = start_price * (1 + upside_cap)

            period_notional = shares * start_price
            period_length = end - start

            for t in range(start + 1, end + 1):
                current_price = underlying_paths[t, path_idx]

                floor_value = max(floor_strike - current_price, 0) * shares
                cap_value = max(current_price - cap_strike, 0) * shares

                elapsed = t - start
                carry_cost = period_notional * carry_annual * (elapsed / trading_days_per_year)

                marked_overlay_value = floor_value - cap_value - carry_cost

                strategy_values[t, path_idx] = max(
                    0.01,
                    shares * current_price + cash + marked_overlay_value
                )

            end_price = underlying_paths[end, path_idx]
            period_return = end_price / start_price - 1

            floor_payoff = max(floor_strike - end_price, 0) * shares
            cap_payment = max(end_price - cap_strike, 0) * shares
            carry_cost = period_notional * carry_annual * (period_length / trading_days_per_year)

            net_cash_flow = floor_payoff - cap_payment - carry_cost
            cash += net_cash_flow

            if cash > 0 and period_return <= redeploy_trigger:
                dollars_to_redeploy = cash * redeploy_rate
                shares += dollars_to_redeploy / end_price
                cash -= dollars_to_redeploy

            if cash < 0:
                dollars_needed = -cash
                shares_to_sell = min(shares * 0.999, dollars_needed / end_price)

                shares -= shares_to_sell
                cash += shares_to_sell * end_price

                if cash < 0:
                    cash = 0.0

            strategy_values[end, path_idx] = max(
                0.01,
                shares * end_price + cash
            )

            start = end

    return strategy_values


hedge_paths = simulate_hedge_strategy_from_underlying(
    underlying_paths=hot_paths,
    reset_days=HEDGE_RESET_DAYS,
    floor_loss=HEDGE_FLOOR_LOSS,
    upside_cap=HEDGE_UPSIDE_CAP,
    carry_annual=HEDGE_CARRY_ANNUAL,
    redeploy_trigger=HEDGE_REDEPLOY_TRIGGER,
    redeploy_rate=HEDGE_REDEPLOY_RATE,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Bottom-decile cohort selection
# ============================================================

def bottom_decile_indices(paths, bottom_decile=0.10):
    final_values = paths[-1, :]
    cutoff = np.percentile(final_values, bottom_decile * 100)
    return np.where(final_values <= cutoff)[0], cutoff


hedge_bottom_indices, hedge_bottom_cutoff = bottom_decile_indices(
    hedge_paths,
    BOTTOM_DECILE
)

hot_bottom_indices, hot_bottom_cutoff = bottom_decile_indices(
    hot_paths,
    BOTTOM_DECILE
)

display_rng = np.random.default_rng(SEED + 999)

hedge_display_indices = display_rng.choice(
    hedge_bottom_indices,
    size=min(N_DISPLAY_PATHS, len(hedge_bottom_indices)),
    replace=False
)

hot_display_indices = display_rng.choice(
    hot_bottom_indices,
    size=min(N_DISPLAY_PATHS, len(hot_bottom_indices)),
    replace=False
)

# ============================================================
# Conditional arithmetic / geometric growth trends
# ============================================================

def conditional_ag_trends(paths, selected_indices, s0=100):
    """
    Conditional A/G trends for a cohort of paths.

    Arithmetic trend:
        Cross-sectional arithmetic mean value through time.

    Geometric trend:
        Cross-sectional geometric mean value through time.

    The visual gap between the two is the volatility-drag story:
        arithmetic outcomes can look acceptable while geometric wealth collapses.
    """

    cohort = paths[:, selected_indices]

    arithmetic_trend = np.nanmean(cohort, axis=1)

    safe_cohort = np.maximum(cohort, 1e-12)
    geometric_trend = s0 * np.exp(
        np.nanmean(np.log(safe_cohort / s0), axis=1)
    )

    return arithmetic_trend, geometric_trend


hedge_cond_arith_path, hedge_cond_geo_path = conditional_ag_trends(
    hedge_paths,
    hedge_bottom_indices,
    INITIAL_INDEX_VALUE
)

hot_cond_arith_path, hot_cond_geo_path = conditional_ag_trends(
    hot_paths,
    hot_bottom_indices,
    INITIAL_INDEX_VALUE
)

# ============================================================
# Conditional summaries and stats
# ============================================================

def summarize_selected_paths(paths, selected_indices):
    cohort = paths[:, selected_indices]

    return {
        "p10": np.percentile(cohort, 10, axis=1),
        "p50": np.percentile(cohort, 50, axis=1),
        "p90": np.percentile(cohort, 90, axis=1),
        "mean_mc": cohort.mean(axis=1)
    }


hedge_bottom_summary = summarize_selected_paths(
    hedge_paths,
    hedge_bottom_indices
)

hot_bottom_summary = summarize_selected_paths(
    hot_paths,
    hot_bottom_indices
)


def calc_path_mdds(paths):
    running_max = np.maximum.accumulate(paths, axis=0)
    drawdowns = paths / running_max - 1
    max_drawdowns = drawdowns.min(axis=0)

    return -max_drawdowns


def calc_realized_vol_drag(paths, selected_indices, trading_days_per_year=252):
    cohort = paths[:, selected_indices]

    daily_returns = cohort[1:, :] / cohort[:-1, :] - 1
    daily_returns = np.where(np.isfinite(daily_returns), daily_returns, np.nan)
    daily_returns = np.clip(daily_returns, -0.999999, None)

    path_arith_growth = np.nanmean(daily_returns, axis=0) * trading_days_per_year
    path_geo_growth = np.nanmean(np.log1p(daily_returns), axis=0) * trading_days_per_year

    path_vol_drag = path_arith_growth - path_geo_growth

    return np.nanmean(path_vol_drag)


def calc_conditioned_stats(paths, selected_indices, years, s0=100):
    cohort = paths[:, selected_indices]
    final_values = cohort[-1, :]

    path_cagrs = (final_values / s0) ** (1 / years) - 1
    path_mdds = calc_path_mdds(cohort)

    stats = {
        "avg_cagr": np.nanmean(path_cagrs),
        "avg_mdd": np.nanmean(path_mdds),
        "vol_drag": calc_realized_vol_drag(paths, selected_indices, TRADING_DAYS_PER_YEAR),
        "terminal_1pct_value": np.percentile(final_values, 1),
        "mean_final": np.nanmean(final_values),
        "median_final": np.nanmedian(final_values),
        "p10_final": np.percentile(final_values, 10),
        "p90_final": np.percentile(final_values, 90),
        "bottom_cutoff": np.percentile(paths[-1, :], BOTTOM_DECILE * 100),
    }

    return stats


hedge_bottom_stats = calc_conditioned_stats(
    paths=hedge_paths,
    selected_indices=hedge_bottom_indices,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

hot_bottom_stats = calc_conditioned_stats(
    paths=hot_paths,
    selected_indices=hot_bottom_indices,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

principal_color = "#777777"

below_principal_color = "#ff4444"
above_principal_color = "#ffd84d"

arith_color = "#33aaff"
geo_color = "#ff66cc"

band_color_hedge = "rgba(0,255,136,0.12)"
band_color_hot = "rgba(255,170,51,0.12)"

percent_bar_color = "#33aaff"
value_bar_color = "#ff6666"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# ============================================================
# Bar chart data
# ============================================================

percent_stat_labels = [
    "Bottom 10%<br>Avg. CAGR",
    "Bottom 10%<br>Avg. MDD",
    "Bottom 10%<br>Vol Drag"
]

value_stat_labels = [
    "Bottom 10%<br>1% Value"
]

def percent_values_from_stats(stats):
    return [
        stats["avg_cagr"] * 100,
        stats["avg_mdd"] * 100,
        stats["vol_drag"] * 100
    ]

def value_values_from_stats(stats):
    return [
        stats["terminal_1pct_value"]
    ]

def percent_text(values):
    return [f"{v:.1f}%" for v in values]

def value_text(values):
    return [f"{v:.0f}" for v in values]

hedge_percent_values = percent_values_from_stats(hedge_bottom_stats)
hot_percent_values = percent_values_from_stats(hot_bottom_stats)

hedge_value_values = value_values_from_stats(hedge_bottom_stats)
hot_value_values = value_values_from_stats(hot_bottom_stats)

all_percent_values = hedge_percent_values + hot_percent_values
all_value_values = hedge_value_values + hot_value_values

percent_axis_min = min(0, min(all_percent_values) * 1.25)
percent_axis_max = max(10, max(all_percent_values) * 1.25)

value_axis_min = 0
value_axis_max = max(100, max(all_value_values) * 1.35)

# ============================================================
# Plot helpers
# ============================================================

def split_path_by_principal(path, principal=100):
    below = np.where(path < principal, path, np.nan)
    above = np.where(path >= principal, path, np.nan)

    return below, above


def make_bottom_decile_title(name, stats):
    return (
        f"<b>{name}: Bottom 10% Terminal Paths</b><br>"
        f"Avg CAGR: {stats['avg_cagr']:.1%} · "
        f"Avg MDD: {stats['avg_mdd']:.1%} · "
        f"Vol Drag: {stats['vol_drag']:.1%}<br>"
        f"Bottom-decile cutoff: {stats['bottom_cutoff']:.0f} · "
        f"1% Value inside cohort: {stats['terminal_1pct_value']:.0f}"
    )


hedge_subtitle = make_bottom_decile_title(
    HEDGE_NAME,
    hedge_bottom_stats
)

hot_subtitle = make_bottom_decile_title(
    HOT_STOCK_NAME,
    hot_bottom_stats
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.50, 0.50],
    row_heights=[0.68, 0.32],
    horizontal_spacing=0.10,
    vertical_spacing=0.18,
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": True}, {"secondary_y": True}]
    ],
    subplot_titles=(
        hedge_subtitle,
        hot_subtitle,
        f"{HEDGE_NAME}: Bottom 10% Metrics",
        f"{HOT_STOCK_NAME}: Bottom 10% Metrics"
    )
)

animated_y_series = []

def add_animated_trace(row, col, y_series, trace):
    fig.add_trace(trace, row=row, col=col)
    animated_y_series.append(y_series)

# ============================================================
# Top row: Hedge bottom-decile paths
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=principal_color, width=1, dash="dash"),
    opacity=0.80,
    row=1,
    col=1
)

add_animated_trace(
    1,
    1,
    hedge_bottom_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_bottom_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HEDGE_NAME} Bottom P90"
    )
)

add_animated_trace(
    1,
    1,
    hedge_bottom_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_bottom_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hedge,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HEDGE_NAME} Bottom 10% Band"
    )
)

for path_num, path_idx in enumerate(hedge_display_indices, start=1):
    y_path = hedge_paths[:, path_idx]
    y_below, y_above = split_path_by_principal(y_path, INITIAL_INDEX_VALUE)

    add_animated_trace(
        1,
        1,
        y_below,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_below[0]],
            mode="lines",
            line=dict(color=below_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HEDGE_NAME} Below Principal Path {path_num}"
        )
    )

    add_animated_trace(
        1,
        1,
        y_above,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_above[0]],
            mode="lines",
            line=dict(color=above_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HEDGE_NAME} Above Principal Path {path_num}"
        )
    )

add_animated_trace(
    1,
    1,
    hedge_cond_arith_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_cond_arith_path[0]],
        mode="lines",
        line=dict(color=arith_color, width=4),
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom-Decile Arithmetic Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% arithmetic trend: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    1,
    hedge_cond_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_cond_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom-Decile Geometric Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% geometric trend: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Top row: Hot stock bottom-decile paths
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=principal_color, width=1, dash="dash"),
    opacity=0.80,
    row=1,
    col=2
)

add_animated_trace(
    1,
    2,
    hot_bottom_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_bottom_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} Bottom P90"
    )
)

add_animated_trace(
    1,
    2,
    hot_bottom_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_bottom_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hot,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} Bottom 10% Band"
    )
)

for path_num, path_idx in enumerate(hot_display_indices, start=1):
    y_path = hot_paths[:, path_idx]
    y_below, y_above = split_path_by_principal(y_path, INITIAL_INDEX_VALUE)

    add_animated_trace(
        1,
        2,
        y_below,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_below[0]],
            mode="lines",
            line=dict(color=below_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HOT_STOCK_NAME} Below Principal Path {path_num}"
        )
    )

    add_animated_trace(
        1,
        2,
        y_above,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_above[0]],
            mode="lines",
            line=dict(color=above_principal_color, width=1.5),
            opacity=0.90,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HOT_STOCK_NAME} Above Principal Path {path_num}"
        )
    )

add_animated_trace(
    1,
    2,
    hot_cond_arith_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_cond_arith_path[0]],
        mode="lines",
        line=dict(color=arith_color, width=4),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom-Decile Arithmetic Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% arithmetic trend: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    2,
    hot_cond_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_cond_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom-Decile Geometric Trend",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Bottom 10% geometric trend: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Bottom row: conditioned metric bar charts
# ============================================================

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hedge_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hedge_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom 10% Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hedge_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hedge_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HEDGE_NAME} Bottom 10% 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=True
)

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hot_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hot_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom 10% Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hot_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hot_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Bottom 10% 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=True
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

n_top_traces = len(animated_y_series)

frame_indices = list(range(1, len(sim_dates), FRAME_STRIDE))

if frame_indices[-1] != len(sim_dates) - 1:
    frame_indices.append(len(sim_dates) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    x_slice = sim_dates[:i + 1]

    frame_data = []

    for y_series in animated_y_series:
        frame_data.append(
            go.Scatter(
                x=x_slice,
                y=y_series[:i + 1]
            )
        )

    frame_data.extend([
        go.Bar(
            x=percent_stat_labels,
            y=hedge_percent_values,
            text=percent_text(hedge_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hedge_value_values,
            text=value_text(hedge_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=percent_stat_labels,
            y=hot_percent_values,
            text=percent_text(hot_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hot_value_values,
            text=value_text(hot_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        )
    ])

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(n_top_traces + 4)),
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": sim_dates[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Bottom 10% Paths: Volatility Drag, Arithmetic Growth, and Geometric Wealth Destruction",
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=950,
    width=1200,
    margin=dict(t=170, b=150, r=80, l=75),
    showlegend=False,
    hovermode="closest",
    barmode="group",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": ANIMATION_REDRAW},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=12))

# ============================================================
# Axes: top row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=TOP_Y_RANGE,
    title_text="Bottom 10% Simulated Value, Start = 100"
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=TOP_Y_RANGE,
    title_text="Bottom 10% Simulated Value, Start = 100"
)

# ============================================================
# Axes: bottom row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Bottom 10% Percent Metrics",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="",
    showgrid=False
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=2,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Bottom 10% Percent Metrics",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="",
    showgrid=False
)

fig.update_traces(
    cliponaxis=False,
    selector=dict(type="bar")
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("bottom_decile_vol_drag_hedge_vs_hot.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### But the First has a Higher CAGR?!  I Still Want to Stock Pick!

Oh man...that was just a passive buy and hold comparison - something your wealth manager would do while ripping you 1% of AUM.

If you want to try to maximize short-term portfolio growth don't expect good long run outcomes.

This is worse than gambling, at least with gambling you know the odds are against you.

I've just shown you the same is true in the context of stock picking, you just didn't know it.

The big difference between the two strategies is volatility drag.

What happens when you layer drawdown monetization on passive buy and hold?  Now we significantly outpreform.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_vrt_daily_returns_10y.csv"

TRADING_DAYS_PER_YEAR = 252
SIM_YEARS = 10
SIM_DAYS = SIM_YEARS * TRADING_DAYS_PER_YEAR

INITIAL_INDEX_VALUE = 100

N_SIM_PATHS = 5000
N_DISPLAY_PATHS = 8

SEED = 42

FRAME_STRIDE = 21
FRAME_DURATION = 30
ANIMATION_REDRAW = False

RISK_FREE_RATE = 0.02

HOT_STOCK_BETA = 2.5
HOT_STOCK_NAME = "2.5 Beta Hot Stock"
HEDGE_NAME = "Hedge"

TOP_Y_RANGE = [0, 1000]

FIT_START_DATE = None
FIT_END_DATE = None

USE_RISK_FREE_CAPM_DRIFT = True

# ============================================================
# Hedge strategy parameters
# ============================================================

# Strategy reset frequency
HEDGE_RESET_DAYS = 63

# Per-reset downside floor.
# Example: 0.10 means the overlay begins monetizing after a -10% reset-period decline.
HEDGE_FLOOR_LOSS = 0.10

# Per-reset upside give-up level.
# Example: 0.28 means gains above +28% during the reset period are given up.
HEDGE_UPSIDE_CAP = 0.28

# Small annualized implementation drag. Keep low for a clean conceptual demo.
HEDGE_CARRY_ANNUAL = 0.00

# Redeploy monetized hedge gains only after meaningful selloffs.
HEDGE_REDEPLOY_TRIGGER = -0.05

# Fraction of positive hedge cash used to buy more shares after a selloff.
HEDGE_REDEPLOY_RATE = 1.00

# ============================================================
# Load data
# ============================================================

df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

required_cols = ["date", "spy_close"]

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV is missing required columns: {missing_cols}")

fit_df = df.copy()

if FIT_START_DATE is not None:
    fit_df = fit_df[fit_df["date"] >= FIT_START_DATE]

if FIT_END_DATE is not None:
    fit_df = fit_df[fit_df["date"] <= FIT_END_DATE]

fit_df = fit_df.reset_index(drop=True)

# ============================================================
# GBM parameter estimation
# ============================================================

def fit_gbm_params_from_prices(data, price_col, trading_days_per_year=252):
    asset_df = data[["date", price_col]].dropna().copy()
    asset_df = asset_df.sort_values("date").reset_index(drop=True)

    if len(asset_df) < 3:
        raise ValueError(f"Not enough price observations for {price_col}")

    asset_df["simple_return"] = asset_df[price_col].pct_change()
    asset_df["log_return"] = np.log(asset_df[price_col] / asset_df[price_col].shift(1))

    returns_df = asset_df.dropna(subset=["simple_return", "log_return"]).copy()

    daily_log_mean = returns_df["log_return"].mean()
    daily_log_var = returns_df["log_return"].var(ddof=1)

    annual_log_growth = daily_log_mean * trading_days_per_year
    annual_variance = daily_log_var * trading_days_per_year
    annual_vol = np.sqrt(annual_variance)

    mu_annual = annual_log_growth + 0.5 * annual_variance
    vol_drag = 0.5 * annual_variance

    params = {
        "name": price_col,
        "price_col": price_col,
        "start_date": asset_df["date"].iloc[0],
        "end_date": asset_df["date"].iloc[-1],
        "n_obs": len(returns_df),

        "daily_log_mean": daily_log_mean,
        "daily_log_variance": daily_log_var,

        "mu_annual": mu_annual,
        "annual_log_growth": annual_log_growth,
        "annual_variance": annual_variance,
        "annual_vol": annual_vol,
        "vol_drag": vol_drag,

        "arithmetic_expected_annual_return": np.exp(mu_annual) - 1,
        "geometric_expected_annual_return": np.exp(annual_log_growth) - 1,
    }

    return params, returns_df


spy_params, spy_returns_df = fit_gbm_params_from_prices(
    fit_df,
    "spy_close",
    TRADING_DAYS_PER_YEAR
)

# ============================================================
# Generate 2.5 beta hot stock strategy from SPY parameters
# ============================================================

def make_beta_strategy_params(
    base_params,
    beta=2.5,
    risk_free_rate=0.02,
    use_capm_drift=True,
    name="2.5 Beta Hot Stock"
):
    spy_mu = base_params["mu_annual"]
    spy_sigma = base_params["annual_vol"]

    if use_capm_drift:
        hot_mu = risk_free_rate + beta * (spy_mu - risk_free_rate)
    else:
        hot_mu = beta * spy_mu

    hot_sigma = beta * spy_sigma
    hot_variance = hot_sigma ** 2
    hot_vol_drag = 0.5 * hot_variance
    hot_annual_log_growth = hot_mu - hot_vol_drag

    hot_params = {
        "name": name,
        "beta": beta,
        "start_date": base_params["start_date"],
        "end_date": base_params["end_date"],
        "n_obs": base_params["n_obs"],

        "mu_annual": hot_mu,
        "annual_log_growth": hot_annual_log_growth,
        "annual_variance": hot_variance,
        "annual_vol": hot_sigma,
        "vol_drag": hot_vol_drag,

        "arithmetic_expected_annual_return": np.exp(hot_mu) - 1,
        "geometric_expected_annual_return": np.exp(hot_annual_log_growth) - 1,
    }

    return hot_params


hot_params = make_beta_strategy_params(
    base_params=spy_params,
    beta=HOT_STOCK_BETA,
    risk_free_rate=RISK_FREE_RATE,
    use_capm_drift=USE_RISK_FREE_CAPM_DRIFT,
    name=HOT_STOCK_NAME
)

# ============================================================
# GBM simulation
# ============================================================

def simulate_gbm_paths(
    mu_annual,
    sigma_annual,
    years=10,
    trading_days_per_year=252,
    n_paths=5000,
    s0=100,
    seed=42
):
    rng = np.random.default_rng(seed)

    n_days = years * trading_days_per_year
    dt = 1 / trading_days_per_year

    z = rng.normal(size=(n_days, n_paths))

    log_returns = (
        (mu_annual - 0.5 * sigma_annual**2) * dt
        + sigma_annual * np.sqrt(dt) * z
    )

    log_paths = np.vstack([
        np.zeros(n_paths),
        np.cumsum(log_returns, axis=0)
    ])

    paths = s0 * np.exp(log_paths)

    return paths, log_returns


hot_paths, hot_sim_log_returns = simulate_gbm_paths(
    mu_annual=hot_params["mu_annual"],
    sigma_annual=hot_params["annual_vol"],
    years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    n_paths=N_SIM_PATHS,
    s0=INITIAL_INDEX_VALUE,
    seed=SEED + 1
)

last_data_date = spy_params["end_date"]

sim_dates = pd.bdate_range(
    start=last_data_date + pd.offsets.BDay(1),
    periods=SIM_DAYS + 1
)

t_years = np.arange(SIM_DAYS + 1) / TRADING_DAYS_PER_YEAR

# ============================================================
# Hedge strategy simulation
# ============================================================

def simulate_hedge_strategy_from_underlying(
    underlying_paths,
    reset_days=63,
    floor_loss=0.10,
    upside_cap=0.28,
    carry_annual=0.00,
    redeploy_trigger=-0.05,
    redeploy_rate=1.00,
    trading_days_per_year=252,
    s0=100
):
    """
    Systematic Hedge framework:

    1. Start with exposure to the 2.5-beta underlying.
    2. At every reset date, define:
        - a downside floor below the current underlying level
        - an upside give-up level above the current underlying level
    3. During selloffs, the floor creates monetizable cash.
    4. When a reset period ends below the redeploy trigger,
       positive hedge cash is used to buy more shares.
    5. During strong rallies, upside above the cap is given up.
    """

    n_steps, n_paths = underlying_paths.shape
    strategy_values = np.zeros_like(underlying_paths)

    for path_idx in range(n_paths):
        shares = s0 / underlying_paths[0, path_idx]
        cash = 0.0

        strategy_values[0, path_idx] = s0

        start = 0

        while start < n_steps - 1:
            end = min(start + reset_days, n_steps - 1)

            start_price = underlying_paths[start, path_idx]
            floor_strike = start_price * (1 - floor_loss)
            cap_strike = start_price * (1 + upside_cap)

            period_notional = shares * start_price
            period_length = end - start

            for t in range(start + 1, end + 1):
                current_price = underlying_paths[t, path_idx]

                floor_value = max(floor_strike - current_price, 0) * shares
                cap_value = max(current_price - cap_strike, 0) * shares

                elapsed = t - start
                carry_cost = period_notional * carry_annual * (elapsed / trading_days_per_year)

                marked_overlay_value = floor_value - cap_value - carry_cost

                strategy_values[t, path_idx] = max(
                    0.01,
                    shares * current_price + cash + marked_overlay_value
                )

            end_price = underlying_paths[end, path_idx]
            period_return = end_price / start_price - 1

            floor_payoff = max(floor_strike - end_price, 0) * shares
            cap_payment = max(end_price - cap_strike, 0) * shares
            carry_cost = period_notional * carry_annual * (period_length / trading_days_per_year)

            net_cash_flow = floor_payoff - cap_payment - carry_cost
            cash += net_cash_flow

            # Monetize drawdown: buy more shares when the overlay pays during a selloff.
            if cash > 0 and period_return <= redeploy_trigger:
                dollars_to_redeploy = cash * redeploy_rate
                shares += dollars_to_redeploy / end_price
                cash -= dollars_to_redeploy

            # If upside give-up creates a cash shortfall, fund it by trimming shares.
            if cash < 0:
                dollars_needed = -cash
                shares_to_sell = min(shares * 0.999, dollars_needed / end_price)

                shares -= shares_to_sell
                cash += shares_to_sell * end_price

                if cash < 0:
                    cash = 0.0

            strategy_values[end, path_idx] = max(
                0.01,
                shares * end_price + cash
            )

            start = end

    return strategy_values


hedge_paths = simulate_hedge_strategy_from_underlying(
    underlying_paths=hot_paths,
    reset_days=HEDGE_RESET_DAYS,
    floor_loss=HEDGE_FLOOR_LOSS,
    upside_cap=HEDGE_UPSIDE_CAP,
    carry_annual=HEDGE_CARRY_ANNUAL,
    redeploy_trigger=HEDGE_REDEPLOY_TRIGGER,
    redeploy_rate=HEDGE_REDEPLOY_RATE,
    trading_days_per_year=TRADING_DAYS_PER_YEAR,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Pick paths to display
# ============================================================

display_rng = np.random.default_rng(SEED + 999)

display_path_indices = display_rng.choice(
    np.arange(N_SIM_PATHS),
    size=N_DISPLAY_PATHS,
    replace=False
)

# ============================================================
# Simulation summaries
# ============================================================

def summarize_paths(paths):
    return {
        "p10": np.percentile(paths, 10, axis=1),
        "p50": np.percentile(paths, 50, axis=1),
        "p90": np.percentile(paths, 90, axis=1),
        "mean_mc": paths.mean(axis=1)
    }


hedge_summary = summarize_paths(hedge_paths)
hot_summary = summarize_paths(hot_paths)


def theoretical_paths(params, t_years, s0=100):
    mu = params["mu_annual"]
    sigma = params["annual_vol"]

    arithmetic_mean_path = s0 * np.exp(mu * t_years)
    geometric_median_path = s0 * np.exp((mu - 0.5 * sigma**2) * t_years)

    return arithmetic_mean_path, geometric_median_path


hot_mean_path, hot_geo_path = theoretical_paths(
    hot_params,
    t_years,
    INITIAL_INDEX_VALUE
)

hedge_mean_path = hedge_summary["mean_mc"]
hedge_median_path = hedge_summary["p50"]

# ============================================================
# Statistics for fixed bottom bar charts
# ============================================================

def calc_path_mdds(paths):
    running_max = np.maximum.accumulate(paths, axis=0)
    drawdowns = paths / running_max - 1
    max_drawdowns = drawdowns.min(axis=0)

    return -max_drawdowns


def calc_realized_vol_drag(paths, trading_days_per_year=252):
    daily_returns = paths[1:, :] / paths[:-1, :] - 1
    daily_returns = np.where(np.isfinite(daily_returns), daily_returns, np.nan)
    daily_returns = np.clip(daily_returns, -0.999999, None)

    arithmetic_growth = np.nanmean(daily_returns) * trading_days_per_year
    geometric_growth = np.nanmean(np.log1p(daily_returns)) * trading_days_per_year

    return arithmetic_growth - geometric_growth


def calc_simulation_stats(paths, years, s0=100):
    final_values = paths[-1, :]

    path_cagrs = (final_values / s0) ** (1 / years) - 1
    path_mdds = calc_path_mdds(paths)

    stats = {
        "avg_cagr": np.nanmean(path_cagrs),
        "avg_mdd": np.nanmean(path_mdds),
        "vol_drag": calc_realized_vol_drag(paths, TRADING_DAYS_PER_YEAR),
        "terminal_1pct_value": np.percentile(final_values, 1),
        "mean_final": np.nanmean(final_values),
        "median_final": np.nanmedian(final_values),
        "p10_final": np.percentile(final_values, 10),
        "p90_final": np.percentile(final_values, 90),
    }

    return stats


hedge_stats = calc_simulation_stats(
    paths=hedge_paths,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

hot_stats = calc_simulation_stats(
    paths=hot_paths,
    years=SIM_YEARS,
    s0=INITIAL_INDEX_VALUE
)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

mean_color = "#33aaff"
geo_color = "#ff6666"

band_color_hedge = "rgba(0,255,136,0.16)"
band_color_hot = "rgba(255,170,51,0.16)"

path_color_hedge = "rgba(224,224,224,0.28)"
path_color_hot = "rgba(224,224,224,0.28)"

baseline_color = "#777777"

percent_bar_color = "#33aaff"
value_bar_color = "#ff6666"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# ============================================================
# Bottom chart data
# ============================================================

percent_stat_labels = [
    "Avg.<br>CAGR",
    "Avg.<br>MDD",
    "Volatility<br>Drag"
]

value_stat_labels = [
    "1%<br>Value"
]

def percent_values_from_stats(stats):
    return [
        stats["avg_cagr"] * 100,
        stats["avg_mdd"] * 100,
        stats["vol_drag"] * 100
    ]

def value_values_from_stats(stats):
    return [
        stats["terminal_1pct_value"]
    ]

def percent_text(values):
    return [f"{v:.1f}%" for v in values]

def value_text(values):
    return [f"{v:.0f}" for v in values]

hedge_percent_values = percent_values_from_stats(hedge_stats)
hot_percent_values = percent_values_from_stats(hot_stats)

hedge_value_values = value_values_from_stats(hedge_stats)
hot_value_values = value_values_from_stats(hot_stats)

all_percent_values = hedge_percent_values + hot_percent_values
all_value_values = hedge_value_values + hot_value_values

percent_axis_min = min(0, min(all_percent_values) * 1.20)
percent_axis_max = max(10, max(all_percent_values) * 1.20)

value_axis_min = 0
value_axis_max = max(100, max(all_value_values) * 1.35)

# ============================================================
# Subplot titles
# ============================================================

def make_hedge_title(stats):
    return (
        f"<b>{HEDGE_NAME}</b><br>"
        f"Floor: -{HEDGE_FLOOR_LOSS:.0%} · "
        f"Redeploy trigger: {HEDGE_REDEPLOY_TRIGGER:.0%}<br>"
        f"Avg CAGR: {stats['avg_cagr']:.1%} · "
        f"Avg MDD: {stats['avg_mdd']:.1%} · "
        f"1% Value: {stats['terminal_1pct_value']:.0f}"
    )


def make_hot_title(params, stats):
    return (
        f"<b>{HOT_STOCK_NAME}</b> · Beta: {HOT_STOCK_BETA:.1f}<br>"
        f"μ: {params['mu_annual']:.1%} · "
        f"σ: {params['annual_vol']:.1%} · "
        f"Vol drag: {stats['vol_drag']:.1%}<br>"
        f"Geo growth: {params['annual_log_growth']:.1%}"
    )


hedge_subtitle = make_hedge_title(hedge_stats)
hot_subtitle = make_hot_title(hot_params, hot_stats)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.50, 0.50],
    row_heights=[0.68, 0.32],
    horizontal_spacing=0.10,
    vertical_spacing=0.18,
    specs=[
        [{"secondary_y": False}, {"secondary_y": False}],
        [{"secondary_y": True}, {"secondary_y": True}]
    ],
    subplot_titles=(
        hedge_subtitle,
        hot_subtitle,
        f"{HEDGE_NAME} Metrics",
        f"{HOT_STOCK_NAME} Metrics"
    )
)

animated_y_series = []

def add_animated_trace(row, col, y_series, trace):
    fig.add_trace(trace, row=row, col=col)
    animated_y_series.append(y_series)

# ============================================================
# Top row: Hedge strategy traces
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

add_animated_trace(
    1,
    1,
    hedge_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HEDGE_NAME} P90"
    )
)

add_animated_trace(
    1,
    1,
    hedge_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(0,255,136,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hedge,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HEDGE_NAME} 10th-90th Percentile Band"
    )
)

for path_num, path_idx in enumerate(display_path_indices, start=1):
    y_path = hedge_paths[:, path_idx]

    add_animated_trace(
        1,
        1,
        y_path,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_path[0]],
            mode="lines",
            line=dict(color=path_color_hedge, width=1),
            opacity=0.85,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HEDGE_NAME} Path {path_num}"
        )
    )

add_animated_trace(
    1,
    1,
    hedge_mean_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_mean_path[0]],
        mode="lines",
        line=dict(color=mean_color, width=4),
        showlegend=False,
        name=f"{HEDGE_NAME} Mean Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Mean value: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    1,
    hedge_median_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hedge_median_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HEDGE_NAME} Median Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Median value: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Top row: Unhedged 2.5 beta strategy traces
# ============================================================

fig.add_hline(
    y=INITIAL_INDEX_VALUE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

add_animated_trace(
    1,
    2,
    hot_summary["p90"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_summary["p90"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} P90"
    )
)

add_animated_trace(
    1,
    2,
    hot_summary["p10"],
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_summary["p10"][0]],
        mode="lines",
        line=dict(color="rgba(255,170,51,0.0)", width=0),
        fill="tonexty",
        fillcolor=band_color_hot,
        showlegend=False,
        hoverinfo="skip",
        name=f"{HOT_STOCK_NAME} 10th-90th Percentile Band"
    )
)

for path_num, path_idx in enumerate(display_path_indices, start=1):
    y_path = hot_paths[:, path_idx]

    add_animated_trace(
        1,
        2,
        y_path,
        go.Scatter(
            x=[sim_dates[0]],
            y=[y_path[0]],
            mode="lines",
            line=dict(color=path_color_hot, width=1),
            opacity=0.85,
            showlegend=False,
            hoverinfo="skip",
            name=f"{HOT_STOCK_NAME} Path {path_num}"
        )
    )

add_animated_trace(
    1,
    2,
    hot_mean_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_mean_path[0]],
        mode="lines",
        line=dict(color=mean_color, width=4),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Arithmetic Mean Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Arithmetic expected value: %{y:.2f}<extra></extra>"
        )
    )
)

add_animated_trace(
    1,
    2,
    hot_geo_path,
    go.Scatter(
        x=[sim_dates[0]],
        y=[hot_geo_path[0]],
        mode="lines",
        line=dict(color=geo_color, width=4, dash="dash"),
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Geometric / Median Path",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Geometric / median value: %{y:.2f}<extra></extra>"
        )
    )
)

# ============================================================
# Bottom row: fixed metric bar charts
# ============================================================

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hedge_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hedge_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HEDGE_NAME} Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hedge_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hedge_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HEDGE_NAME} 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=1,
    secondary_y=True
)

fig.add_trace(
    go.Bar(
        x=percent_stat_labels,
        y=hot_percent_values,
        marker=dict(color=percent_bar_color, opacity=0.85),
        text=percent_text(hot_percent_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} Percent Metrics",
        hovertemplate="%{x}<br>%{y:.2f}%<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=value_stat_labels,
        y=hot_value_values,
        marker=dict(color=value_bar_color, opacity=0.85),
        text=value_text(hot_value_values),
        textposition="outside",
        showlegend=False,
        name=f"{HOT_STOCK_NAME} 1% Value",
        hovertemplate="%{x}<br>%{y:.2f}<extra></extra>"
    ),
    row=2,
    col=2,
    secondary_y=True
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

n_top_traces = len(animated_y_series)

frame_indices = list(range(1, len(sim_dates), FRAME_STRIDE))

if frame_indices[-1] != len(sim_dates) - 1:
    frame_indices.append(len(sim_dates) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    x_slice = sim_dates[:i + 1]

    frame_data = []

    for y_series in animated_y_series:
        frame_data.append(
            go.Scatter(
                x=x_slice,
                y=y_series[:i + 1]
            )
        )

    frame_data.extend([
        go.Bar(
            x=percent_stat_labels,
            y=hedge_percent_values,
            text=percent_text(hedge_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hedge_value_values,
            text=value_text(hedge_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=percent_stat_labels,
            y=hot_percent_values,
            text=percent_text(hot_percent_values),
            textposition="outside",
            marker=dict(color=percent_bar_color, opacity=0.85),
            showlegend=False
        ),
        go.Bar(
            x=value_stat_labels,
            y=hot_value_values,
            text=value_text(hot_value_values),
            textposition="outside",
            marker=dict(color=value_bar_color, opacity=0.85),
            showlegend=False
        )
    ])

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(n_top_traces + 4)),
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": sim_dates[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text="Hedge vs. 2.5-Beta Hot Stock Strategy: Monetizing Drawdowns Into Higher Exposure",
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=950,
    width=1200,
    margin=dict(t=155, b=150, r=80, l=75),
    showlegend=False,
    hovermode="closest",
    barmode="group",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": FRAME_DURATION, "redraw": ANIMATION_REDRAW},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": ANIMATION_REDRAW},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=13))

# ============================================================
# Axes: top row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=TOP_Y_RANGE,
    title_text="Simulated Value, Start = 100"
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[sim_dates[0], sim_dates[-1]],
    title_text="Simulation Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=TOP_Y_RANGE,
    title_text="Simulated Value, Start = 100"
)

# ============================================================
# Axes: bottom row
# ============================================================

fig.update_xaxes(
    axis_style,
    row=2,
    col=1,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Percent",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=1,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="1% Value",
    showgrid=False
)

fig.update_xaxes(
    axis_style,
    row=2,
    col=2,
    tickfont=dict(color=off_white, size=10),
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=False,
    range=[percent_axis_min, percent_axis_max],
    title_text="Percent",
    ticksuffix="%"
)

fig.update_yaxes(
    axis_style,
    row=2,
    col=2,
    secondary_y=True,
    range=[value_axis_min, value_axis_max],
    title_text="1% Value",
    showgrid=False
)

fig.update_traces(
    cliponaxis=False,
    selector=dict(type="bar")
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("hedge_vs_2p5_beta_hot_stock_animation.html", include_plotlyjs="cdn")

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
 - This notebook explores why relying on individual stock selection is generally a flawed approach for building long-term portfolio wealth.
 - Empirical analysis demonstrates that the majority of single stocks underperform market indexes over extended horizons, with a small subset driving most of the overall market's returns.
 - Visualizations highlight how focusing on individual winners is akin to "lottery ticket" investing—most picks lag the broad market, and missing the rare outliers can be severely detrimental.
 - Instead, diversified portfolio construction—holding a broad mix of assets—consistently reduces risk and leads to more stable compounded returns over time.
 - The key message: persistent outperformance by selecting the "right" stocks is exceedingly rare and extremely difficult to repeat. Embracing diversification, rather than stock picking, is a much more robust strategy for long-term growth.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$